# 07 - Executive insights

Notebook tong hop insight cuoi cung cho bao cao/slides bang tieng Viet.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
EXPORT_DIR = ROOT / 'dashboard' / 'exports'
PROCESSED_DIR = ROOT / 'data' / 'processed'

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 140)


def read_csv(relative_path):
    path = ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(f'Missing required input: {path}')
    return pd.read_csv(path)


def pct(numerator, denominator):
    return np.where(pd.Series(denominator).replace(0, np.nan).notna(), numerator / denominator * 100, np.nan)

overview = read_csv('dashboard/exports/vw_executive_overview.csv').iloc[0]
posts = read_csv('data/processed/posts.csv')
comments = read_csv('data/processed/comments.csv')
viral = read_csv('dashboard/exports/vw_viral_posts.csv')
heatmap = read_csv('dashboard/exports/vw_posting_time_heatmap.csv')
sentiment = read_csv('dashboard/exports/vw_sentiment_trend.csv')
competitors = read_csv('dashboard/exports/vw_competitor_benchmark.csv')


In [ ]:
brand_name = 'Highlands Coffee Vietnam'
total_reach = competitors['total_reach'].sum()
total_engagement = competitors['total_engagement'].sum()
brand = competitors[competitors['page_name'].eq(brand_name)].iloc[0]
best_brand_post = posts[posts['page_name'].eq(brand_name)].sort_values('engagement_rate', ascending=False).iloc[0]
best_heat_rate = heatmap.dropna(subset=['avg_engagement_rate']).sort_values(['avg_engagement_rate', 'total_engagement'], ascending=False).iloc[0]
best_heat_volume = heatmap.sort_values('total_engagement', ascending=False).iloc[0]
insights = pd.DataFrame([
    {
        'theme': 'Baseline',
        'key_finding': 'YouTube da co baseline du dung cho dieu hanh nhung engagement rate trung binh con thap.',
        'metrics': f"{int(overview['total_posts'])} posts; {overview['total_reach']/1_000_000:.2f}M reach; {overview['avg_engagement_rate']:.3f}% avg ER",
        'recommendation': 'Theo doi baseline nay sau moi refresh va dat muc tieu tang ER truoc khi mo rong kenh.'
    },
    {
        'theme': 'Content',
        'key_finding': 'Mot so video don le dang keo phan lon reach va engagement cua toan bo tap du lieu.',
        'metrics': f"Top post {viral.iloc[0]['reach']/1_000_000:.2f}M reach; {int(viral.iloc[0]['engagement_count']):,} engagements; Highlands top ER {best_brand_post['engagement_rate']:.2f}%",
        'recommendation': 'Phan tich motif cua top video va ap dung vao lich noi dung Highlands co CTA ro hon.'
    },
    {
        'theme': 'Posting time',
        'key_finding': 'Khung gio tot nhat theo ER khac voi khung gio tao nhieu engagement nhat.',
        'metrics': f"Best ER: {best_heat_rate['day_name']} {int(best_heat_rate['hour_of_day']):02d}:00 = {best_heat_rate['avg_engagement_rate']:.2f}%; best volume: {best_heat_volume['day_name']} {int(best_heat_volume['hour_of_day']):02d}:00 = {int(best_heat_volume['total_engagement']):,} engagements",
        'recommendation': 'Chay A/B lich dang giua slot ER cao va slot volume cao trong 4 tuan.'
    },
    {
        'theme': 'Sentiment',
        'key_finding': 'Sentiment hien chu yeu la neutral, negative thap nhung co mot diem can xu ly.',
        'metrics': f"{int(sentiment['comment_count'].sum())} comments; {sentiment['positive_count'].sum()/sentiment['comment_count'].sum()*100:.2f}% positive; {sentiment['negative_count'].sum()/sentiment['comment_count'].sum()*100:.2f}% negative",
        'recommendation': 'Tang cau hoi mo trong caption/video va phan hoi comment negative lien quan trai nghiem cua hang.'
    },
    {
        'theme': 'SOV',
        'key_finding': 'Highlands co reach SOV lon nhung engagement SOV thap hon muc reach SOV.',
        'metrics': f"Highlands reach SOV {brand['total_reach']/total_reach*100:.2f}%; engagement SOV {brand['total_engagement']/total_engagement*100:.2f}%; {int(brand['post_count'])} posts",
        'recommendation': 'Do lai SOV theo owned brand vs creator/industry de co benchmark canh tranh chinh xac hon.'
    },
])
display(insights)


In [ ]:
for _, row in insights.iterrows():
    print(f"## {row['theme']}")
    print(f"- Key finding: {row['key_finding']}")
    print(f"- Supporting data: {row['metrics']}")
    print(f"- Recommended action: {row['recommendation']}")
    print()


## Insight

- Key finding: Ban dieu hanh nen doc dashboard theo 5 lop: baseline, content, posting time, sentiment va SOV.
- Supporting data: 876 posts, 167.40M reach, 53.85K engagements trong giai doan 2017-09-26 den 2026-05-31.
- Recommended action: Dung cac insight tren lam phan ket luan chinh trong bao cao BI va cap nhat khi export moi duoc sinh ra.